In [4]:
!pip3 install -q litellm langchain langchain-community langchain-openai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [7]:
import sys
!{sys.executable} -m pip install -q litellm langchain langchain-community langchain-openai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [10]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
from litellm import completion
import litellm

In [11]:
litellm.suppress_debug_info = True

In [12]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [16]:
from dotenv import load_dotenv
import os
load_dotenv()
print("OpenAI key loaded:", "YES" if os.getenv("OPENAI_API_KEY") else "NO")
print("Gemini key loaded:", "YES" if os.getenv("GOOGLE_API_KEY") else "NO")
print("Groq key loaded:", "YES" if os.getenv("GROQ_API_KEY") else "NO")

OpenAI key loaded: NO
Gemini key loaded: YES
Groq key loaded: YES


In [17]:
# SIMPLE LITE LLM EXAMPLE - UNIFIED API

In [23]:
from litellm import completion

# ------------------ OPEN AI ----------------------
# response_openai = completion(
#     model = "gpt-4o-mini",
#     messages=[{"role" : "user", "content": "Explain CLAUDE in one sentence."}]
# )
# print("OpenAI", response_openai.choices[0].message.content)

# -------------------- GROQ ------------------------
response_groq = completion(
    model = "groq/llama-3.3-70b-versatile",
    messages=[{"role" : "user", "content": "Explain CLAUDE in one sentence."}]
)
print("--GROQ--", response_groq.choices[0].message.content)


--GROQ-- CLAUDE (Conversational Large language model Application with Driven Exploration) is an artificial intelligence chatbot developed by Anthropic, designed to engage in natural-sounding conversations, answer questions, and provide information on a wide range of topics while prioritizing safety and user experience.


In [24]:
from litellm import completion
prompt = "Explain CLAUDE FABLE and why US govt banned it. Only in 20 words excatly."
providers = [
    ("--- OpenAI ---",     "gpt-4o-mini"),
    ("--- Groq ---",       "groq/llama-3.3-70b-versatile"),
    ("--- Anthropic ---",  "claude-3-5-haiku-20241022"),
    ("--- Gemini ---",     "gemini/gemini-1.5-flash"),
]

# ONE FUNCTION CALL FOR MULTIPLE PROVIDERS
for label, model in providers:
    try:
        x = completion(model = model, messages = [{"role":"user", "content":prompt}])
        print(f"{label:<15}: {x.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: --- FAILED --- {type(e).__name__}")

--- OpenAI --- : --- FAILED --- AuthenticationError
--- Groq ---   : CLAUDE FABLE: AI chatbot, banned in US due to misinformation concerns allegedly.
--- Anthropic ---: --- FAILED --- BadRequestError
--- Gemini --- : --- FAILED --- NotFoundError


In [25]:
# AUTOMATIC FALLBACK -- WHEN ANY MODEL GOES DOWN

In [ ]:
# WHENEVER MODEL WE HAD SELECTED IS FAILED AS BELOW "gpt-4o-mini" IS NOT WORKING THEN 
# IT'LL GO AND SELECT THE MODEL PRESENT IN THE FALLBACKS 
from litellm import completion
response = completion(
    model = "gpt-4o-mini",
    messages= [{"role":"user", "content": "Explain LLM Guardrails in 30 words"}],
    fallbacks=[
        "groq/llama-3.3-70b-versatile",
        "gemini/gemini-1.5-flash"
    ]
)
print("Response:", response.choices[0].message.content[:200],"\n")
print("Model Used", response.model)

18:59:40 - LiteLLM:ERROR: fallback_utils.py:68 - Fallback attempt failed for model gpt-4o-mini: litellm.AuthenticationError: AuthenticationError: OpenAIException - The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/litellm/llms/openai/openai.py", line 904, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(  # type: ignore
                                  ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
        is_async=True,
        ^^^^^^^^^^^^^^
    ...<7 lines>...
        shared_session=shared_session,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/litellm/llms/openai/openai.py", line 386, in _get_openai_client
    _new_client: Union[OpenAI, AsyncOpenAI] = AsyncOpenAI(
       

Response: LLM Guardrails: Safety features and guidelines that regulate Large Language Model outputs to prevent misinformation, bias, and harmful content. ...

Model Used llama-3.3-70b-versatile


In [ ]:
# TRACKING THE COST OF LLM GATEWAYS -- AUTOMATICALLY CALCULATES THE COST OF EVERY CALL

In [31]:
from litellm import completion, completion_cost
response = completion(
    model = "groq/llama-3.3-70b-versatile",
    messages = [{"role":"user", "content": "Write about Sukhoi 30 MKI in 40 words."}]
)
cost = completion_cost(completion_response = response, model="groq/llama-3.3-70b-versatile")
print("Response: \n", response.choices[0].message.content)
print("Input Tokens: \n", response.usage.prompt_tokens)
print("Output Tokens: \n", response.usage.completion_tokens)
print(f"Cost:    ${cost:.8f}")


Response: 
 The Sukhoi Su-30 MKI is a multirole fighter jet used by the Indian Air Force, known for its advanced avionics, maneuverability, and versatility in air-to-air and air-to-ground combat operations.
Input Tokens: 
 49
Output Tokens: 
 47
Cost:    $0.00006604


In [32]:
# CACHING - AVOID PAYING TWICE FOR SAME QUESTION

In [33]:
import litellm

litellm.callbacks = []
litellm.success_callback  = []
litellm.faliure_callback = []
litellm._async_success_callback = []
litellm._async_faliure_callback = []
litellm.cache = None
print("LITELLM STATE RESET\n")

LITELLM STATE RESET

